# 00 — Datasets: raw files, unified schema, counts and label mappings

**Fake News & Misinformation Detector** (MSE1 · Dataset criterion). This notebook

1. loads every **raw** file in `data/raw/` (fetched by `make download`, no Kaggle needed) and prints row counts + label distributions,
2. loads the **processed** parquet files produced by `make data` (`src/preprocess/`), shows the unified schema, counts, label distributions, split sizes and the first rows,
3. prints the label-mapping tables (LIAR 6 → 5 / 3, FEVER → stance) and the download manifest (URL, SHA-256, date).

Expected sizes (master doc §10): WELFake 72,134 · ISOT 44,898 · LIAR 12,836 · FEVER subset 20k + 3k · FakeNewsNet-PolitiFact ~1,056.
Runtime ≈ 1 min on CPU.

In [1]:
import json, sys, platform, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config as C
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 160)
print("python", platform.python_version(), "| pandas", pd.__version__)
print("raw:", C.RAW_DIR, "| processed:", C.PROCESSED_DIR, "| splits:", C.SPLITS_DIR)
t0 = time.time()

python 3.11.11 | pandas 2.3.3
raw: /home/nikhil/Desktop/nlp/data/raw | processed: /home/nikhil/Desktop/nlp/data/processed | splits: /home/nikhil/Desktop/nlp/data/splits


## 1. Raw files (as downloaded)

In [2]:
EXPECTED = {"welfake": 72_134, "isot": 44_898, "liar": 12_836, "fever_train_full": 145_449,
            "fever_dev_full": 19_998, "fnn_politifact": 1_056}
raw = {}
raw["welfake"] = pd.read_csv(C.RAW_DIR / "welfake" / "WELFake_Dataset.csv")
isot_true = pd.read_csv(C.RAW_DIR / "isot" / "True.csv"); isot_fake = pd.read_csv(C.RAW_DIR / "isot" / "Fake.csv")
raw["isot"] = pd.concat([isot_true.assign(label="real"), isot_fake.assign(label="fake")], ignore_index=True)
from src.preprocess.load import LIAR_COLUMNS
raw["liar"] = pd.concat([pd.read_csv(C.RAW_DIR / "liar" / f, sep="\t", header=None, names=LIAR_COLUMNS, quoting=3,
                                     dtype=str, keep_default_na=False).assign(split=s)
                         for f, s in (("train.tsv", "train"), ("valid.tsv", "val"), ("test.tsv", "test"))], ignore_index=True)
raw["fever_train_full"] = pd.read_json(C.RAW_DIR / "fever" / "train.jsonl", lines=True)
raw["fever_dev_full"] = pd.read_json(C.RAW_DIR / "fever" / "shared_task_dev.jsonl", lines=True)
raw["fnn_politifact"] = pd.concat([pd.read_csv(C.RAW_DIR / "fnn_politifact" / f, dtype=str).assign(label=l)
                                   for f, l in (("politifact_fake.csv", "fake"), ("politifact_real.csv", "real"))], ignore_index=True)
rows = [{"dataset": k, "rows": len(v), "expected": EXPECTED[k], "match": len(v) == EXPECTED[k], "columns": ", ".join(map(str, v.columns[:6]))}
        for k, v in raw.items()]
pd.DataFrame(rows)

,dataset,rows,expected,match,columns
0,welfake,72134,72134,True,"Unnamed: 0, title, text, label"
1,isot,44898,44898,True,"title, text, subject, date, label"
2,liar,12836,12836,True,"id, label, statement, subject, speaker, speaker_job"
3,fever_train_full,145449,145449,True,"id, verifiable, label, claim, evidence"
4,fever_dev_full,19998,19998,True,"id, verifiable, label, claim, evidence"
5,fnn_politifact,1056,1056,True,"id, news_url, title, tweet_ids, label"


### 1.1 Raw label distributions

> **WELFake label quirk.** The Zenodo/Kaggle description says `0 = fake, 1 = real`, but the file is the other way round:
> label **0** rows (35,028) are the *real* articles (60 % carry a Reuters dateline) and label **1** rows (37,106) are the *fake* ones
> (`Featured image via …`, `[VIDEO]`, ALL-CAPS titles). The paper's own counts (35,028 real / 37,106 fake) confirm it, so
> `src/preprocess/load.py` maps `0 → real`, `1 → fake`.

In [3]:
w = raw["welfake"].copy(); w["text"] = w["text"].fillna(""); w["title"] = w["title"].fillna("")
print("WELFake raw label counts:", w["label"].value_counts().sort_index().to_dict())
print(pd.DataFrame({"label": [0, 1],
                    "frac_with_(Reuters)": [w[w.label == l]["text"].str.contains(r"\(Reuters\)").mean().round(3) for l in (0, 1)],
                    "frac_title_has_[VIDEO]": [w[w.label == l]["title"].str.contains(r"\[VIDEO\]").mean().round(3) for l in (0, 1)],
                    "frac_'Featured image'": [w[w.label == l]["text"].str.contains("Featured image").mean().round(3) for l in (0, 1)]}).to_string(index=False))
print("\nISOT:", raw["isot"]["label"].value_counts().to_dict(), "| subjects:", raw["isot"]["subject"].value_counts().to_dict())
print("ISOT dates:", pd.to_datetime(raw["isot"]["date"], errors="coerce", format="mixed").agg(["min", "max"]).dt.date.to_dict())
print("\nLIAR 6-way:", raw["liar"]["label"].value_counts().to_dict(), "| official splits:", raw["liar"]["split"].value_counts().to_dict())
print("\nFEVER train (full):", raw["fever_train_full"]["label"].value_counts().to_dict())
print("FEVER shared_task_dev (full):", raw["fever_dev_full"]["label"].value_counts().to_dict())
print("\nFakeNewsNet PolitiFact:", raw["fnn_politifact"]["label"].value_counts().to_dict())

WELFake raw label counts: {0: 35028, 1: 37106}


 label  frac_with_(Reuters)  frac_title_has_[VIDEO]  frac_'Featured image'
     0                0.607                   0.000                    0.0
     1                0.000                   0.068                    0.2

ISOT: {'fake': 23481, 'real': 21417} | subjects: {'politicsNews': 11272, 'worldnews': 10145, 'News': 9050, 'politics': 6841, 'left-news': 4459, 'Government News': 1570, 'US_News': 783, 'Middle-east': 778}
ISOT dates: {'min': datetime.date(2015, 3, 31), 'max': datetime.date(2018, 2, 19)}

LIAR 6-way: {'half-true': 2638, 'false': 2511, 'mostly-true': 2466, 'barely-true': 2108, 'true': 2063, 'pants-fire': 1050} | official splits: {'train': 10269, 'val': 1284, 'test': 1283}

FEVER train (full): {'SUPPORTS': 80035, 'NOT ENOUGH INFO': 35639, 'REFUTES': 29775}
FEVER shared_task_dev (full): {'NOT ENOUGH INFO': 6666, 'SUPPORTS': 6666, 'REFUTES': 6666}

FakeNewsNet PolitiFact: {'real': 624, 'fake': 432}


### 1.2 Three raw rows per dataset

In [4]:
for k in ("welfake", "isot", "liar", "fever_train_full", "fnn_politifact"):
    print(f"--- {k} ---")
    cols = [c for c in raw[k].columns if c not in ("tweet_ids",)][:6]
    display(raw[k][cols].head(3))

--- welfake ---


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #Bla...,No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #Bla...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PRO...,"Now, most of the demonstrators gathered last night were exercising their constitution...",1


--- isot ---


,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip their fiscal script",WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congr...,politicsNews,"December 31, 2017",real
1,U.S. military to accept transgender recruits on Monday: Pentagon,WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist...,politicsNews,"December 29, 2017",real
2,Senior U.S. Republican senator: 'Let Mr. Mueller do his job',WASHINGTON (Reuters) - The special counsel investigation of links between Russia and P...,politicsNews,"December 31, 2017",real


--- liar ---


,id,label,statement,subject,speaker,speaker_job
0,2635.json,false,Says the Annies List political group supports third-trimester abortions on demand.,abortion,dwayne-bohac,State representative
1,10540.json,half-true,When did the decline of coal start? It started when natural gas took off that started ...,"energy,history,job-accomplishments",scott-surovell,State delegate
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by voting to give George Bush the benefit of ...",foreign-policy,barack-obama,President


--- fever_train_full ---


,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [92206, 104971, Fox_Broadcasting_Company,..."
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271, 187499, Roman_Atwood, 3]]]"
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, sculpture, music, painting, poetry litera...","[[[255136, 254645, History_of_art, 2]]]"


--- fnn_politifact ---


,id,news_url,title,label
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs,fake
1,politifact15156,politics2020.info/index.php/2018/03/13/court-orders-obama-to-pay-400-million-in-restit...,Court Orders Obama To Pay $400 Million In Restitution,fake
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467344/update-second-roy-moore-accuser-work...,UPDATE: Second Roy Moore Accuser Works For Michelle Obama Right NOW -,fake


## 2. Processed data (`make data` → `data/processed/*.parquet`)

Unified schema for **every** dataset (`src/config.SCHEMA`):

| column | meaning |
|---|---|
| `id` | deterministic unique id (`welfake_12`, `isot_real_7`, `liar_2635`, `fever_75397`, `fnn_fake_politifact14984`) |
| `title` | headline (None for LIAR / FEVER) |
| `text` | cased, cleaned text (article body / statement / claim / headline) |
| `label` | dataset-native label in project vocabulary: `fake`/`real`; LIAR 6-way; FEVER stance `SUPPORTS`/`REFUTES`/`NEUTRAL` |
| `label5` | project 5-class mapping (`REAL` / `FAKE` / `PARTIALLY TRUE` / `MISLEADING`); None for FEVER |
| `label3` | 3-way collapse (`TRUE` / `MIXED` / `FALSE`); None for FEVER |
| `source_dataset` | `welfake` · `isot` · `liar` · `fever_subset` · `fnn_politifact` |
| `date` | ISO publication date where available (ISOT only) |
| `split` | `train` / `val` / `test` (stratified 80/10/10 seed 42; LIAR official; FEVER train/dev subset) |
| `meta` | JSON string with dataset-specific extras (ISOT subject, LIAR speaker/party/context, FEVER evidence pages, FNN url) |

In [5]:
proc = {n: pd.read_parquet(C.PROCESSED_DIR / f"{n}.parquet") for n in C.DATASETS if (C.PROCESSED_DIR / f"{n}.parquet").exists()}
summary = pd.DataFrame([{"dataset": n, "rows": len(d), "columns_ok": list(d.columns) == C.SCHEMA,
                         "splits": d["split"].value_counts().to_dict(), "label": d["label"].value_counts().to_dict()}
                        for n, d in proc.items()])
summary

,dataset,rows,columns_ok,splits,label
0,welfake,60799,True,"{'train': 48639, 'val': 6080, 'test': 6080}","{'real': 34295, 'fake': 26504}"
1,isot,37567,True,"{'train': 30053, 'val': 3757, 'test': 3757}","{'real': 20905, 'fake': 16662}"
2,liar,12836,True,"{'train': 10269, 'val': 1284, 'test': 1283}","{'half-true': 2638, 'false': 2511, 'mostly-true': 2466, 'barely-true': 2108, 'true': 2..."
3,fever_subset,23000,True,"{'train': 20000, 'val': 3000}","{'REFUTES': 7667, 'NEUTRAL': 7667, 'SUPPORTS': 7666}"
4,fnn_politifact,1056,True,{},"{'real': 624, 'fake': 432}"


### 2.1 Schema table (identical across datasets)

In [6]:
schema_rows = []
for n, d in proc.items():
    for c in d.columns:
        schema_rows.append({"dataset": n, "column": c, "dtype": str(d[c].dtype), "non_null": int(d[c].notna().sum())})
sch = pd.DataFrame(schema_rows).pivot(index="column", columns="dataset", values="non_null").reindex(C.SCHEMA)
sch

dataset,fever_subset,fnn_politifact,isot,liar,welfake
column,,,,,
id,23000,1056,37567,12836,60799
title,0,1056,37567,0,60493
text,23000,1056,37567,12836,60799
label,23000,1056,37567,12836,60799
label5,0,1056,37567,12836,60799
label3,0,1056,37567,12836,60799
source_dataset,23000,1056,37567,12836,60799
date,0,0,37566,0,0
split,23000,0,37567,12836,60799


### 2.2 Counts: raw → processed, per dataset (see `docs/mse1_make_data.log` for the drop/dedup breakdown)

In [7]:
summ = json.loads((C.PROCESSED_DIR / "make_data_summary.json").read_text())
cols = ["dataset", "rows_in", "dropped_empty", "dropped_short", "exact_dups_removed", "near_dups_removed", "rows_out", "seconds"]
pd.DataFrame([{c: s.get(c, 0) for c in cols} for s in summ])

,dataset,rows_in,dropped_empty,dropped_short,exact_dups_removed,near_dups_removed,rows_out,seconds
0,welfake,72134,1591,1047,8217,480,60799,238.6
1,isot,44898,1439,478,5380,34,37567,165.2
2,liar,12836,0,0,0,0,12836,0.2
3,fever_subset,165447,0,0,10388,0,23000,1.3
4,fnn_politifact,1056,0,0,0,0,1056,0.0


### 2.3 Label distributions (label / label5 / label3) and split sizes

In [8]:
for n, d in proc.items():
    print(f"=== {n}: {len(d):,} rows ===")
    print("  label :", d["label"].value_counts().to_dict())
    print("  label5:", d["label5"].value_counts(dropna=False).to_dict())
    print("  label3:", d["label3"].value_counts(dropna=False).to_dict())
    print("  split :", d["split"].value_counts(dropna=False).to_dict())
    ct = pd.crosstab(d["split"], d["label"], normalize="index").round(3)
    display(ct)

=== welfake: 60,799 rows ===
  label : {'real': 34295, 'fake': 26504}
  label5: {'REAL': 34295, 'FAKE': 26504}
  label3: {'TRUE': 34295, 'FALSE': 26504}
  split : {'train': 48639, 'val': 6080, 'test': 6080}


label,fake,real
split,,
test,0.436,0.564
train,0.436,0.564
val,0.436,0.564


=== isot: 37,567 rows ===
  label : {'real': 20905, 'fake': 16662}
  label5: {'REAL': 20905, 'FAKE': 16662}
  label3: {'TRUE': 20905, 'FALSE': 16662}
  split : {'train': 30053, 'val': 3757, 'test': 3757}


label,fake,real
split,,
test,0.443,0.557
train,0.444,0.556
val,0.443,0.557


=== liar: 12,836 rows ===
  label : {'half-true': 2638, 'false': 2511, 'mostly-true': 2466, 'barely-true': 2108, 'true': 2063, 'pants-fire': 1050}
  label5: {'REAL': 4529, 'FAKE': 3561, 'PARTIALLY TRUE': 2638, 'MISLEADING': 2108}
  label3: {'MIXED': 4746, 'TRUE': 4529, 'FALSE': 3561}
  split : {'train': 10269, 'val': 1284, 'test': 1283}


label,barely-true,false,half-true,mostly-true,pants-fire,true
split,,,,,,
test,0.167,0.195,0.208,0.194,0.072,0.164
train,0.161,0.195,0.207,0.191,0.082,0.164
val,0.185,0.205,0.193,0.195,0.090,0.132


=== fever_subset: 23,000 rows ===
  label : {'REFUTES': 7667, 'NEUTRAL': 7667, 'SUPPORTS': 7666}
  label5: {<NA>: 23000}
  label3: {<NA>: 23000}
  split : {'train': 20000, 'val': 3000}


label,NEUTRAL,REFUTES,SUPPORTS
split,,,
train,0.333,0.333,0.333
val,0.333,0.333,0.333


=== fnn_politifact: 1,056 rows ===
  label : {'real': 624, 'fake': 432}
  label5: {'REAL': 624, 'FAKE': 432}
  label3: {'TRUE': 624, 'FALSE': 432}
  split : {<NA>: 1056}


label
split


### 2.4 Split ID files (`data/splits/{welfake,isot,liar}_{train,val,test}.csv`)

In [9]:
rows = []
for n in C.SPLIT_DATASETS:
    total = sum(len(pd.read_csv(C.SPLITS_DIR / f"{n}_{s}.csv")) for s in C.SPLIT_NAMES)
    for s in C.SPLIT_NAMES:
        k = len(pd.read_csv(C.SPLITS_DIR / f"{n}_{s}.csv"))
        rows.append({"dataset": n, "split": s, "rows": k, "fraction": round(k / total, 4)})
pd.DataFrame(rows).pivot(index="dataset", columns="split", values=["rows", "fraction"])

rows                  fraction           
split      test    train     val     test train  val
dataset                                             
isot     3757.0  30053.0  3757.0      0.1   0.8  0.1
liar     1283.0  10269.0  1284.0      0.1   0.8  0.1
welfake  6080.0  48639.0  6080.0      0.1   0.8  0.1

### 2.5 First rows of each processed table

In [10]:
for n, d in proc.items():
    print(f"--- {n} ---")
    display(d.drop(columns=["meta"]).head(3).assign(text=lambda x: x["text"].str.slice(0, 120) + "…"))

--- welfake ---


,id,title,text,label,label5,label3,source_dataset,date,split
0,welfake_0,LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #Bla...,No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #Bla...,fake,FAKE,FALSE,welfake,<NA>,train
1,welfake_10,GOP Senator Just Smacked Down The Most Punchable Alt-Right Nazi On The Internet,The most punchable Alt-Right Nazi on the internet just got a thorough beatdown from Se...,fake,FAKE,FALSE,welfake,<NA>,train
2,welfake_100,One person shot in Portland as anti-Trump protesters cross bridge: police,One person was shot at an anti-Trump demonstration in Portland on Saturday as proteste...,real,REAL,TRUE,welfake,<NA>,train


--- isot ---


,id,title,text,label,label5,label3,source_dataset,date,split
0,isot_fake_0,Donald Trump Sends Out Embarrassing New Year's Eve Message; This is Disturbing,Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. I...,fake,FAKE,FALSE,isot,2017-12-31,train
1,isot_fake_1,Drunk Bragging Trump Staffer Started Russian Collusion Investigation,House Intelligence Committee Chairman Devin Nunes is going to have a bad day. He s bee...,fake,FAKE,FALSE,isot,2017-12-31,train
2,isot_fake_10,"Papa John's Founder Retires, Figures Out Racism Is Bad For Business","A centerpiece of Donald Trump s campaign, and now his presidency, has been his white s...",fake,FAKE,FALSE,isot,2017-12-21,train


--- liar ---


,id,title,text,label,label5,label3,source_dataset,date,split
0,liar_1,<NA>,The attorney general requires that rape victims pay for the rape kit.…,pants-fire,FAKE,FALSE,liar,<NA>,train
1,liar_100,<NA>,"""He's sued gun manufacturers. He was supportive of Brady. He was supportive of things ...",true,REAL,TRUE,liar,<NA>,train
2,liar_1000,<NA>,"In the 1970s, the swine flu broke out . . . under another Democrat, President Jimmy Ca...",pants-fire,FAKE,FALSE,liar,<NA>,train


--- fever_subset ---


,id,title,text,label,label5,label3,source_dataset,date,split
0,fever_10003,<NA>,Chris Froome is not a cyclist.…,REFUTES,<NA>,<NA>,fever_subset,<NA>,train
1,fever_100032,<NA>,Lewis Hamilton was named Food Personality of the Year.…,NEUTRAL,<NA>,<NA>,fever_subset,<NA>,train
2,fever_100052,<NA>,Eminem was not an original member of Soul Intent.…,REFUTES,<NA>,<NA>,fever_subset,<NA>,train


--- fnn_politifact ---


,id,title,text,label,label5,label3,source_dataset,date,split
0,fnn_fake_politifact11773,Virginia Republican Wants Schools To Check Children's Genitals Before Using Bathroom,Virginia Republican Wants Schools To Check Children's Genitals Before Using Bathroom…,fake,FAKE,FALSE,fnn_politifact,<NA>,<NA>
1,fnn_fake_politifact13038,The numbers don't lie... - Occupy Democrats,The numbers don't lie... - Occupy Democrats…,fake,FAKE,FALSE,fnn_politifact,<NA>,<NA>
2,fnn_fake_politifact13467,Mental Images,Mental Images…,fake,FAKE,FALSE,fnn_politifact,<NA>,<NA>


### 2.6 Artefact removal check: `(Reuters)` must be gone from processed ISOT / WELFake

In [11]:
for n in ("isot", "welfake"):
    if n in proc:
        print(n, "rows containing '(Reuters)':", int(proc[n]["text"].str.contains(r"\(Reuters\)").sum()),
              "| 'http':", int(proc[n]["text"].str.contains("http").sum()))
ex = raw["isot"].loc[raw["isot"]["label"] == "real", "text"].iloc[0][:160]
print("\nraw ISOT   :", ex)
print("processed  :", proc["isot"].loc[proc["isot"]["id"] == "isot_real_0", "text"].iloc[0][:160])

isot rows containing '(Reuters)': 0 | 'http': 0


welfake rows containing '(Reuters)': 0 | 'http': 14

raw ISOT   : WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay
processed  : The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called h


## 3. Label mappings (master doc §10.1 / §10.2)

In [12]:
print("LIAR 6 -> project 5-class / 3-way")
display(pd.DataFrame({"liar_label": C.LIAR_LABELS, "label5": [C.LIAR_TO_5[l] for l in C.LIAR_LABELS],
                      "label3": [C.LIAR_TO_3[l] for l in C.LIAR_LABELS]}))
print("Binary datasets (WELFake / ISOT / FakeNewsNet):", C.BINARY_TO_5, C.BINARY_TO_3)
print("FEVER -> stance:", C.FEVER_TO_STANCE)
print("Verdict vocabulary:", C.VERDICTS, "(UNVERIFIABLE arises only from retrieval failure at run time)")

LIAR 6 -> project 5-class / 3-way


,liar_label,label5,label3
0,pants-fire,FAKE,FALSE
1,false,FAKE,FALSE
2,barely-true,MISLEADING,MIXED
3,half-true,PARTIALLY TRUE,MIXED
4,mostly-true,REAL,TRUE
5,true,REAL,TRUE


Binary datasets (WELFake / ISOT / FakeNewsNet): {'real': 'REAL', 'fake': 'FAKE'} {'real': 'TRUE', 'fake': 'FALSE'}
FEVER -> stance: {'SUPPORTS': 'SUPPORTS', 'REFUTES': 'REFUTES', 'NOT ENOUGH INFO': 'NEUTRAL'}
Verdict vocabulary: ['REAL', 'FAKE', 'PARTIALLY TRUE', 'MISLEADING', 'UNVERIFIABLE'] (UNVERIFIABLE arises only from retrieval failure at run time)


## 4. Download manifest (URL · SHA-256 · date) — mirrored in `data/README.md`

In [13]:
man = json.loads((C.RAW_DIR / "manifest.json").read_text())
pd.DataFrame([{"file": k, **v} for k, v in man.items()])[["file", "dataset", "size", "sha256", "downloaded", "url"]]

,file,dataset,size,sha256,downloaded,url
0,fever/shared_task_dev.jsonl,fever,4349935,e89865bfe1b4dd054e03dd57d7241a6fde24862905f31117cf0cd719f7c78df7,2026-08-28,https://fever.ai/download/fever/shared_task_dev.jsonl
1,fever/train.jsonl,fever,33024303,eba7e8f87076753f8494718b9a857827af7bf73e76c9e4b75420207d26e588b6,2026-08-28,https://fever.ai/download/fever/train.jsonl
2,fnn_politifact/politifact_fake.csv,fnn_politifact,3286418,abe7fe7aad801b1e2ab5fc963cbd0881d9670190262ae3df85e91c3380bd004c,2026-08-28,https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_fake.csv
3,fnn_politifact/politifact_real.csv,fnn_politifact,8278658,2500f86a7addca0f59fe8cf089c0f802d3eca6980f53538992b3b0e9687dbae0,2026-08-28,https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_real.csv
4,isot/News-_dataset.zip,isot,43106824,1846d4ca7abfedb53a4f7c233489d23a0e6c8bb79e7f56fb8345c4faceae5268,2026-08-28,https://onlineacademiccommunity.uvic.ca/isot/wp-content/uploads/sites/7295/2023/03/New...
5,liar/liar_dataset.zip,liar,1013571,611c1addad919743dde15822b87a60bfb760d8f85597f25289e34621800654c7,2026-08-28,https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
6,welfake/WELFake_Dataset.csv,welfake,245086152,665331424230fc452e9482c3547a6a199a2c29745ade8d236950d1d105223773,2026-08-28,https://zenodo.org/api/records/4561253/files/WELFake_Dataset.csv/content


## 5. Summary: expected vs. actual

In [14]:
actual = {"welfake": len(proc["welfake"]), "isot": len(proc["isot"]) if "isot" in proc else 0, "liar": len(proc["liar"]),
          "fever_subset": len(proc["fever_subset"]), "fnn_politifact": len(proc["fnn_politifact"])}
exp = {"welfake": "72,134 raw (after dedup: see log)", "isot": "44,898 raw (after dedup: see log)", "liar": "12,836",
       "fever_subset": "23,000 (20k train + 3k dev)", "fnn_politifact": "~1,056"}
display(pd.DataFrame({"expected": exp, "raw_rows": {"welfake": len(raw["welfake"]), "isot": len(raw["isot"]), "liar": len(raw["liar"]),
                                                    "fever_subset": len(raw["fever_train_full"]) + len(raw["fever_dev_full"]),
                                                    "fnn_politifact": len(raw["fnn_politifact"])}, "processed_rows": actual}))
print(f"notebook runtime: {time.time() - t0:.1f}s")

,expected,raw_rows,processed_rows
welfake,"72,134 raw (after dedup: see log)",72134,60799
isot,"44,898 raw (after dedup: see log)",44898,37567
liar,"12,836",12836,12836
fever_subset,"23,000 (20k train + 3k dev)",165447,23000
fnn_politifact,"~1,056",1056,1056


notebook runtime: 7.4s
